**Create Splits and Generate Training Features**

We will define a pipeline to created a pooled embedding representing all user reviews for the user associated with a given review, excluding the review itself. We will do the same thing for the business embedding as well. Star rating will be left in as the training target. There will be 2 versions, one where the business category tag embedding is pooled as if it were another review, and another where it is held as separate model input. This is to test out different approaches to feeding this info into the model. Splits will be 80, 10, 10 between train, validation, and test.

In [ ]:
import json
import numpy as np
from collections import defaultdict

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
def build_pooled_training_files(
    reviews_json: str,
    categories_json: str,
    output_prefix: str,  # e.g. "/content/drive/.../pooled"
    train_ratio: float = 0.8,
    val_ratio: float = 0.1,
    test_ratio: float = 0.1,
):
    """
    Writes train/val/test features for both pooled and separate category tags.
    Outputs:
      {output_prefix}_train_A.json
      {output_prefix}_val_A.json
      {output_prefix}_test_A.json
      {output_prefix}_train_B.json
      {output_prefix}_val_B.json
      {output_prefix}_test_B.json

    Cols:
    - stars, user_pool, business_pool_with_category
    - stars, user_pool, business_pool, business_category_embedding
    """
    def permute_indices(n, user_id):
        rng = np.random.default_rng(seed=hash(user_id) & 0xFFFFFFFF)
        return rng.permutation(n)

    def pooled_excluding_self(emb_list, exclude_pos):
        if len(emb_list) <= 1:
            return np.zeros_like(emb_list[0])
        total = sum(emb_list)
        excl = emb_list[exclude_pos]
        return (total - excl) / (len(emb_list) - 1)

    # Load reviews
    reviews = []
    with open(reviews_json, 'r', encoding='utf-8') as f:
        for line in f:
            r = json.loads(line)
            emb = r.get('text') if 'text' in r else r.get('embedding')
            if emb is None:
                raise ValueError(f"Review {r.get('review_id')} missing embedding field")
            r['emb'] = np.array(emb, dtype=float)
            reviews.append(r)

    if not reviews:
        raise RuntimeError("No reviews loaded.")
    embedding_dim = reviews[0]['emb'].shape[0]

    # Load category embeddings
    business_cat_emb = {}
    with open(categories_json, 'r', encoding='utf-8') as f:
        for line in f:
            b = json.loads(line)
            b_emb = b.get('text') if 'text' in b else b.get('embedding')
            if b_emb is None:
                continue
            business_cat_emb[b['business_id']] = np.array(b_emb, dtype=float)

    def get_category_embedding(business_id):
        return business_cat_emb.get(business_id, np.zeros(embedding_dim, dtype=float))

    # Split per user
    user_to_indices = defaultdict(list)
    for idx, r in enumerate(reviews):
        user_to_indices[r['user_id']].append(idx)

    for user_id, idx_list in user_to_indices.items():
        n = len(idx_list)
        if n == 0:
            continue
        perm = permute_indices(n, user_id)
        train_cut = int(np.floor(train_ratio * n))
        val_cut = train_cut + int(np.floor(val_ratio * n))
        for pos, permuted_pos in enumerate(perm):
            review_idx = idx_list[permuted_pos]
            if pos < train_cut:
                reviews[review_idx]['split'] = 'train'
            elif pos < val_cut:
                reviews[review_idx]['split'] = 'val'
            else:
                reviews[review_idx]['split'] = 'test'

    # Group indices
    user_split_groups = defaultdict(list)
    business_split_groups = defaultdict(list)
    for idx, r in enumerate(reviews):
        user_split_groups[(r['user_id'], r['split'])].append(idx)
        business_split_groups[(r['business_id'], r['split'])].append(idx)

    # Prepare output file paths
    variants = ['train', 'val', 'test']
    handles = {}
    for split in variants:
        handles[(split, 'A')] = open(f"{output_prefix}_{split}_A.json", 'w', encoding='utf-8')
        handles[(split, 'B')] = open(f"{output_prefix}_{split}_B.json", 'w', encoding='utf-8')

    try:
        for idx, r in enumerate(reviews):
            user_id = r['user_id']
            business_id = r['business_id']
            split = r['split']
            stars = r['stars']
            review_id = r['review_id']
            emb = r['emb']

            # user pool
            user_group = user_split_groups.get((user_id, split), [])
            user_embs = [reviews[i]['emb'] for i in user_group]
            if idx in user_group:
                exclude_pos = user_group.index(idx)
                user_pool = pooled_excluding_self(user_embs, exclude_pos)
            else:
                user_pool = np.zeros(embedding_dim, dtype=float)

            # business pool (original reviews)
            biz_group = business_split_groups.get((business_id, split), [])
            biz_embs = [reviews[i]['emb'] for i in biz_group]
            if idx in biz_group:
                exclude_pos_biz = biz_group.index(idx)
                business_pool = pooled_excluding_self(biz_embs, exclude_pos_biz)
            else:
                business_pool = np.zeros(embedding_dim, dtype=float)

            # category
            cat_emb = get_category_embedding(business_id)

            # A: pool business reviews plus category as if it's another review (exclude self only if it's in biz_group)
            if idx in biz_group:
                # include category embedding in the list, exclude the current review
                combined = biz_embs + [cat_emb]
                business_pool_with_category = pooled_excluding_self(combined, exclude_pos_biz)
            else:
                # review isn't in the business group: pool is just the category embedding
                business_pool_with_category = cat_emb

            # round vectors to 4 decimals here
            def round_vec(v):
                return np.round(v, 4).tolist()

            out_A = {
                'review_id': review_id,
                'user_id': user_id,
                'business_id': business_id,
                'split': split,
                'stars': stars,
                'user_pool': round_vec(user_pool),
                'business_pool_with_category': round_vec(business_pool_with_category)
            }

            out_B = {
                'review_id': review_id,
                'user_id': user_id,
                'business_id': business_id,
                'split': split,
                'stars': stars,
                'user_pool': round_vec(user_pool),
                'business_pool': round_vec(business_pool),
                'business_category_embedding': round_vec(cat_emb)
            }

            # write to proper split files
            fa = handles[(split, 'A')]
            fb = handles[(split, 'B')]
            fa.write(json.dumps(out_A) + '\n')
            fb.write(json.dumps(out_B) + '\n')
    finally:
        for h in handles.values():
            h.close()

Execute function to generate model features and training examples for each encoding strategy

In [ ]:
# Generate features for Word2Vec (Word Average Vector)

build_pooled_training_files(
    reviews_json='/content/drive/MyDrive/Colab_Folder/266_Project/Data/Word2Vec/user_tower_reviews.json',
    categories_json='/content/drive/MyDrive/Colab_Folder/266_Project/Data/Word2Vec/business_category_tags.json',
    output_prefix='/content/drive/MyDrive/Colab_Folder/266_Project/Features/Word2Vec/'
)

In [ ]:
# Generate features for SBERT 6 Layer

build_pooled_training_files(
    reviews_json='/content/drive/MyDrive/Colab_Folder/266_Project/Data/SBERT_6Layer/user_tower_reviews.json',
    categories_json='/content/drive/MyDrive/Colab_Folder/266_Project/Data/SBERT_6Layer/business_category_tags.json',
    output_prefix='/content/drive/MyDrive/Colab_Folder/266_Project/Features/SBERT_6Layer/'
)

In [ ]:
# Generate features for SBERT 12 Layer

build_pooled_training_files(
    reviews_json='/content/drive/MyDrive/Colab_Folder/266_Project/Data/SBERT_12Layer/user_tower_reviews.json',
    categories_json='/content/drive/MyDrive/Colab_Folder/266_Project/Data/SBERT_12Layer/business_category_tags.json',
    output_prefix='/content/drive/MyDrive/Colab_Folder/266_Project/Features/SBERT_12Layer/'
)

In [ ]:
# Generate features for JINA

build_pooled_training_files(
    reviews_json='/content/drive/MyDrive/Colab_Folder/266_Project/Data/JINA/user_tower_reviews.json',
    categories_json='/content/drive/MyDrive/Colab_Folder/266_Project/Data/JINA/business_category_tags.json',
    output_prefix='/content/drive/MyDrive/Colab_Folder/266_Project/Features/JINA/'
)